[Guide Index](README.md)


# 05a OWL 2 Manchester Syntax

Manchester Syntax is an English-readable notation for OWL 2 ontologies — `Class: Cat SubClassOf: Animal` reads naturally, rather than as raw RDF triples:

| Manchester Syntax | Equivalent RDF (Turtle) |
|---|---|
| `Class: Cat SubClassOf: Animal` | `:Cat rdf:type owl:Class . :Cat rdfs:subClassOf :Animal .` |

`OxGraph` (the Oxigraph-backed graph class; "OxGraph" is the working name of what becomes `StarLayerGraph`, see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)) parses Manchester Syntax text using `g.parse(data=..., format="manchester")`, and serializes an RDF graph back into Manchester Syntax text using `g.serialize(format="manchester")`.

This guide first covers ordinary parsing and serialization (sections 1-11), then treats a Manchester Syntax document's own internal structure - frames, clauses, expressions - as an inspectable, editable RDF graph in its own right (sections 12-17), the same "syntax-tree-as-RDF" pattern the [SPARQL queries as RDF guide](05b-sparql-query-as-rdf.ipynb) uses for SPARQL query algebra.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). No extra runtime dependencies are needed - Manchester Syntax parsing is pure Python, and `pyshacl` (needed for section 17's validation step) ships as a standard dependency already.

Run cells from top to bottom; later cells reuse variables from earlier ones.

An `OxGraph` starts with no prefix bindings, so the helper below binds the ones it prints with (`:`, `owl:`, `rdfs:`); `serialize(format="turtle", pretty=True)` then writes grouped, nested Turtle with only the prefixes that are used. (`xsd:` is added by the Manchester parser when a document uses it.)


In [1]:
from rdflib import Namespace
from rdflib.namespace import OWL, RDFS
from starlayer.graph.oxgraph import OxGraph

EX = Namespace("http://example.org/vehicles#")


def show(manchester_text):
    """Parse a Manchester Syntax snippet and print the RDF it maps to."""
    g = OxGraph()
    g.bind("", EX)
    g.bind("owl", OWL)
    g.bind("rdfs", RDFS)
    g.parse(data=f"Prefix: : <{EX}>\n" + manchester_text, format="manchester")
    print(g.serialize(format="turtle", pretty=True))
    return g

## 1. Classes: `SubClassOf:`, `SuperClassOf:`, `DisjointWith:`

Each `Class:` frame declares an `owl:Class`; its clauses add axioms about it. `SuperClassOf:` is the reverse-direction spelling of `SubClassOf:` — `Vehicle SuperClassOf: Motorcycle` means exactly the same thing as `Motorcycle SubClassOf: Vehicle`.

In [2]:
_ = show("""
Class: Vehicle
Class: Car
    SubClassOf: Vehicle
Class: Motorcycle
    SubClassOf: Vehicle
Class: Truck
    SubClassOf: Vehicle
    DisjointWith: Car

Class: Vehicle
    SuperClassOf: Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Motorcycle a owl:Class ;
    rdfs:subClassOf :Vehicle .

:Truck a owl:Class ;
    rdfs:subClassOf :Vehicle ;
    owl:disjointWith :Car .

:Vehicle a owl:Class .



## 2. Class expressions: `and` / `or` / `not` / `that`

Note: `that` is a readability synonym for `and`, used for refining a restriction in a sentence-like way ("a family vehicle that has airbags").

In [3]:
_ = show("""
Class: Car
Class: Truck
DataProperty: hasAirbags

Class: ElectricCar
    SubClassOf: Car and (not Truck)

Class: FamilyVehicle
    EquivalentTo: Car or Truck

Class: SafeFamilyVehicle
    EquivalentTo: FamilyVehicle that (hasAirbags value true)
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:ElectricCar a owl:Class ;
    rdfs:subClassOf [
        a owl:Class ;
        owl:intersectionOf (
            :Car
            [
                a owl:Class ;
                owl:complementOf :Truck
            ]
        )
    ] .

:FamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:unionOf ( :Car :Truck )
    ] .

:SafeFamilyVehicle a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:intersectionOf (
            :FamilyVehicle
            [
                a owl:Restriction ;
                owl:hasValue true ;
                owl:onProperty :hasAirbags
            ]
        )
    ] .

:Truck a owl:Class .

:hasAirbags a owl:DatatypeProperty .



## 3. Restrictions: `some` / `only` / `onlysome` / `value` / `Self`, and cardinality

`onlysome` is shorthand for `(p some C) and (p only C)` together. `min`/`max`/`exactly` work with or without a qualifying filler class — compare `SmallCar` (qualified) below to `BigCar` (unqualified).

In [4]:
_ = show("""
Class: Car
Class: Person
ObjectProperty: hasPassenger
ObjectProperty: hasOwner
Individual: Alice
ObjectProperty: monitorsItself
DataProperty: hasDoors

Class: FullCar
    SubClassOf: Car
    SubClassOf: hasPassenger some Person

Class: PassengerOnlyCar
    SubClassOf: Car
    SubClassOf: hasPassenger only Person

Class: BalancedCar
    SubClassOf: Car
    SubClassOf: hasPassenger onlysome Person

Class: AlicesCar
    SubClassOf: Car
    SubClassOf: hasOwner value Alice

Class: SelfMonitoringCar
    SubClassOf: Car
    SubClassOf: monitorsItself Self

Class: Sedan
    SubClassOf: Car
    SubClassOf: hasDoors exactly 4

Class: SmallCar
    SubClassOf: Car
    SubClassOf: hasPassenger max 2 Person

Class: BigCar
    SubClassOf: Car
    SubClassOf: hasPassenger min 5 Person
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:AlicesCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:hasValue :Alice ;
        owl:onProperty :hasOwner
    ] .

:BalancedCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Class ;
        owl:intersectionOf (
            [
                a owl:Restriction ;
                owl:onProperty :hasPassenger ;
                owl:someValuesFrom :Person
            ]
            [
                a owl:Restriction ;
                owl:allValuesFrom :Person ;
                owl:onProperty :hasPassenger
            ]
        )
    ] .

:BigCar a owl:Class ;
    rdfs:subClassOf :Car, [
        a owl:Restriction ;
        owl:minQualifiedCardinality 5 ;
        owl:onClass :Person ;
        owl:onProperty :hasPassenger
    ] .

:Car a owl:Class .

:FullCar a owl:Class 

## 4. Enumerations: `{a, b, c}`

A class defined as exactly the individuals listed — `owl:oneOf`.

In [5]:
_ = show("""
Individual: Red
Individual: Green
Individual: Blue

Class: PrimaryColor
    EquivalentTo: {Red, Green, Blue}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Blue a owl:NamedIndividual .

:Green a owl:NamedIndividual .

:PrimaryColor a owl:Class ;
    owl:equivalentClass [
        a owl:Class ;
        owl:oneOf ( :Red :Green :Blue )
    ] .

:Red a owl:NamedIndividual .



## 5. Object and data properties

`Domain:`/`Range:`, `Characteristics:`, `InverseOf:`, `SubPropertyOf:`/`SuperPropertyOf:`, and `SubPropertyChain:` (object properties only — `hasParent o hasParent` composed into `hasGrandparent`).

In [6]:
_ = show("""
Class: Vehicle
Class: Engine
ObjectProperty: hasEngine
    Domain: Vehicle
    Range: Engine
    Characteristics: Functional

ObjectProperty: poweredBy
    InverseOf: hasEngine

ObjectProperty: hasSubEngine
    SubPropertyOf: hasEngine

ObjectProperty: hasComponent
    SuperPropertyOf: hasSubEngine

ObjectProperty: hasParent
ObjectProperty: hasGrandparent
    SubPropertyChain: hasParent o hasParent

DataProperty: hasWeightKg
    Domain: Vehicle
    Range: xsd:decimal
    Characteristics: Functional
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:Engine a owl:Class .

:Vehicle a owl:Class .

:hasComponent a owl:ObjectProperty .

:hasEngine a owl:FunctionalProperty, owl:ObjectProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range :Engine .

:hasGrandparent a owl:ObjectProperty ;
    owl:propertyChainAxiom ( :hasParent :hasParent ) .

:hasParent a owl:ObjectProperty .

:hasSubEngine a owl:ObjectProperty ;
    rdfs:subPropertyOf :hasComponent, :hasEngine .

:hasWeightKg a owl:DatatypeProperty, owl:FunctionalProperty ;
    rdfs:domain :Vehicle ;
    rdfs:range xsd:decimal .

:poweredBy a owl:ObjectProperty ;
    owl:inverseOf :hasEngine .



## 6. Individuals: `Types:`, `Facts:`, `SameAs:`, `DifferentFrom:`

`Facts:` assertions can be negated with a leading `not` — mapped to `owl:NegativePropertyAssertion`, not silently dropped.

In [7]:
_ = show("""
Class: Car
ObjectProperty: hasOwner
DataProperty: hasVIN
Individual: Alice
Individual: Bob
Individual: mycar2

Individual: mycar
    Types: Car
    Facts: hasOwner Alice, hasVIN "1HGCM82633A004352", not hasOwner Bob
    SameAs: mycar2
    DifferentFrom: Bob
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Alice a owl:NamedIndividual .

:Bob a owl:NamedIndividual .

:Car a owl:Class .

:hasOwner a owl:ObjectProperty .

:hasVIN a owl:DatatypeProperty .

:mycar a :Car, owl:NamedIndividual ;
    :hasOwner :Alice ;
    :hasVIN "1HGCM82633A004352" ;
    owl:differentFrom :Bob ;
    owl:sameAs :mycar2 .

:mycar2 a owl:NamedIndividual .

_:b0 a owl:NegativePropertyAssertion ;
    owl:assertionProperty :hasOwner ;
    owl:sourceIndividual :mycar ;
    owl:targetIndividual :Bob .



## 7. Data ranges and `Datatype:` frames

Facet-restricted datatypes (`xsd:integer[>= 0, <= 400]`), a union of datatypes, and an explicit enumeration of data values (`DataOneOf`).

In [8]:
_ = show("""
Datatype: PositiveInt
    EquivalentTo: xsd:integer[>= 0]

DataProperty: hasSpeedKmh
    Range: xsd:integer[>= 0, <= 400]

DataProperty: hasFuelType
    Range: xsd:string or xsd:integer

DataProperty: hasStatusCode
    Range: {"active", "retired"}
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:PositiveInt a rdfs:Datatype ;
    owl:equivalentClass [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
        )
    ] .

:hasFuelType a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:unionOf ( xsd:string xsd:integer )
    ] .

:hasSpeedKmh a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:onDatatype xsd:integer ;
        owl:withRestrictions (
            [
                xsd:minInclusive 0
            ]
            [
                xsd:maxInclusive 400
            ]
        )
    ] .

:hasStatusCode a owl:DatatypeProperty ;
    rdfs:range [
        a rdfs:Datatype ;
        owl:oneOf ( "active" "retired" )
    ] .



## 8. `HasKey:` and `DisjointUnionOf:`

`HasKey:` says a property (or set of properties) uniquely identifies individuals of a class; `DisjointUnionOf:` says a class is exactly the (pairwise disjoint) union of the listed subclasses.

In [9]:
_ = show("""
Class: Car
Class: Truck
Class: Motorcycle
DataProperty: hasVIN

Class: Vehicle
    HasKey: hasVIN
    DisjointUnionOf: Car, Truck, Motorcycle
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class .

:Motorcycle a owl:Class .

:Truck a owl:Class .

:Vehicle a owl:Class ;
    owl:disjointUnionOf ( :Car :Truck :Motorcycle ) ;
    owl:hasKey ( :hasVIN ) .

:hasVIN a owl:DatatypeProperty .



## 9. Frame-level `Annotations:`

A standalone `Annotations:` clause maps straight to plain triples about the frame's own subject — `rdfs:label`, `rdfs:comment`, or any other annotation property.

In [10]:
_ = show("""
Class: Car
    Annotations: rdfs:label "Car", rdfs:comment "A wheeled motor vehicle"
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Car a owl:Class ;
    rdfs:comment "A wheeled motor vehicle" ;
    rdfs:label "Car" .



### 9.1 Inline per-axiom `Annotations:`

A different, more targeted form: an `Annotations:` block written *inside* another clause's list, immediately before one specific item, annotates just that one axiom — via `owl:Axiom` reification (`owl:annotatedSource`/`owl:annotatedProperty`/`owl:annotatedTarget` plus the annotation itself), alongside the plain axiom triple, never instead of it. The annotation applies to exactly the item it precedes — `Pet` below is asserted as a superclass of `Cat` too, but isn't annotated.

In [11]:
_ = show("""
Class: Animal
Class: Pet
Class: Cat
    SubClassOf: Annotations: rdfs:comment "domesticated" Animal, Pet
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:subClassOf :Animal, :Pet .

:Pet a owl:Class .

_:b0 a owl:Axiom ;
    rdfs:comment "domesticated" ;
    owl:annotatedProperty rdfs:subClassOf ;
    owl:annotatedSource :Cat ;
    owl:annotatedTarget :Animal .



## 10. Top-level Misc axioms

`EquivalentClasses:`/`DisjointClasses:` and their property-flavored counterparts `EquivalentProperties:`/`DisjointProperties:`, plus `SameIndividual:`/`DifferentIndividuals:` — each accepts two or more names at once, not just a pair.

In [12]:
_ = show("""
Class: Auto
Class: Car
Class: Motorcar
EquivalentClasses: Auto, Car, Motorcar

Class: Motorcycle
Class: Bicycle
DisjointClasses: Car, Motorcycle, Bicycle

ObjectProperty: hasOwner
ObjectProperty: ownedBy
EquivalentProperties: hasOwner, ownedBy

ObjectProperty: hasPart
DisjointProperties: hasOwner, hasPart

Individual: mycar
Individual: myauto
SameIndividual: mycar, myauto

Individual: yourcar
DifferentIndividuals: mycar, yourcar
""")

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Auto a owl:Class ;
    owl:equivalentClass :Car .

:Bicycle a owl:Class .

:Car a owl:Class ;
    owl:equivalentClass :Motorcar .

:Motorcar a owl:Class .

:Motorcycle a owl:Class .

:hasOwner a owl:ObjectProperty ;
    owl:equivalentProperty :ownedBy ;
    owl:propertyDisjointWith :hasPart .

:hasPart a owl:ObjectProperty .

:myauto a owl:NamedIndividual .

:mycar a owl:NamedIndividual ;
    owl:differentFrom :yourcar ;
    owl:sameAs :myauto .

:ownedBy a owl:ObjectProperty .

:yourcar a owl:NamedIndividual .

_:b0 a owl:AllDisjointClasses ;
    owl:members ( :Car :Motorcycle :Bicycle ) .



## 11. Going back: RDF → Manchester Syntax

`format="manchester"` works for `serialize()` too - the reverse direction. This is a genuinely different, harder kind of problem than parsing: rather than deterministically producing RDF from text, the serializer has to *recognize* patterns in a flat, unordered triple set (which blank nodes are restrictions, which are `rdf:List` cells, which came from an `owl:Axiom` annotation block) and reconstruct frames from `rdf:type` declarations. `SuperClassOf:`/`onlysome`/small top-level `EquivalentClasses:`-style Misc axioms always come back in their canonical expanded form (see [manchester_syntax_gap_analysis.md](../../packages/starlayer/docs/graph/manchester_syntax_gap_analysis.md) for exactly why) - everything else round-trips losslessly.

In [13]:
g = OxGraph()
g.bind("", EX)
g.parse(data=f"Prefix: : <{EX}>\n" + """
Class: Animal
Class: Cat
    SubClassOf: Animal
    SubClassOf: hasLegs exactly 4
    Annotations: rdfs:label "Cat"
ObjectProperty: hasLegs
""", format="manchester")

print(g.serialize(format="manchester"))

Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :hasLegs exactly 4, :Animal
    Annotations: rdfs:label "Cat"

ObjectProperty: :hasLegs



## 12. A Manchester Syntax document's own structure, as RDF

`starlayer.starontology.manchester` represents a Manchester Syntax document's own internal syntax tree (frames, clauses, class/property expressions, data ranges) as RDF, using the `manch:` ontology - mirroring the sibling `starlayer.sparql` package's `sqe:` treatment of SPARQL query structure (see the [SPARQL queries as RDF guide](05b-sparql-query-as-rdf.ipynb) for that design). This is a separate concern from sections 1-11 above: treating the *document's own structure* as data you can encode, inspect, edit, and validate, rather than just parsing text into OWL RDF or serializing OWL RDF back to text. The `manch:` vocabulary and shapes are shipped as real standalone `.ttl` files (`starlayer/starontology/manchester-ast-ontology.ttl`/`manchester_shapes.ttl`), usable by any external RDF tool with zero Python dependency.

**Fidelity scope**: this matches the parser's own internal AST exactly, the same bar `sqe:` itself holds for SPARQL - `onlysome`/`that`/`SuperClassOf:` spelling desugar the same way they already do at parse time (with one exception: `SuperClassOf:`/`SuperPropertyOf:` carry a `manch:reversed` flag, since - unlike other spelling choices - which keyword was written changes the compiled OWL semantics, not just how it reads).

**Which graph class.** The whole guide uses `OxGraph`. The syntax-tree functions in this part (`manchester_parse_to_tree()`, `manchester_tree_to_text()`, `manchester_tree_to_owl()`, `manchester_validate()`) take an `OxGraph` and give one back: pass `graph=OxGraph()` to `manchester_parse_to_tree()` to build the tree in an `OxGraph` (pyoxigraph terms; without `graph=` it builds the previous rdflib-based graph class, as before), and `manchester_validate()` returns its report as an `OxGraph` for an `OxGraph` tree. `parse_manchester()` is the low-level compiler and still yields rdflib triples, which is why sections 15-16 use it only to count and compare triples.

Everything below is grounded in one small ontology fragment, so later sections' effects show up as a real, different compiled meaning - not just triple counts.


In [14]:
from starlayer.graph.parsers.manchester_parser import parse_manchester

HEADER = 'Prefix: : <http://example.org/vehicles#>\n'

text = HEADER + """
Class: Vehicle
Class: Car
    SubClassOf: Vehicle

ObjectProperty: hasWheelCount
"""

original_triples = parse_manchester(text)
for t in original_triples:
    print(t)

(rdflib.term.URIRef('http://example.org/vehicles#Vehicle'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Class'))
(rdflib.term.URIRef('http://example.org/vehicles#Car'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Class'))
(rdflib.term.URIRef('http://example.org/vehicles#Car'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://example.org/vehicles#Vehicle'))
(rdflib.term.URIRef('http://example.org/vehicles#hasWheelCount'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))


## 13. Parse straight to `manch:` RDF

`manchester_parse_to_tree()` reuses the real parser's own tokenizer/frame/clause/expression grammar completely unchanged - it just captures the syntax tree as RDF via a small set of hooks instead of discarding it after compiling to OWL. Returns just the graph - every graph it produces holds exactly one `manch:Document` node, so there's no separate root to track. Pass `graph=OxGraph()` to have the tree built in an `OxGraph`.

In [15]:
from starlayer.graph.oxgraph import OxGraph
from starlayer.starontology.manchester import manchester_parse_to_tree

graph = manchester_parse_to_tree(text, graph=OxGraph())
print('encoded triples:', len(graph))
print(graph.serialize(format='turtle', pretty=True, only_used=True))


encoded triples:

 32
@prefix manch: <https://github.com/hidden-graph/starlayer/ns/graph/manchester-ast#> .

_:b0 a manch:Document ;
    manch:frames (
        [
            a manch:ClassFrame ;
            manch:clauses () ;
            manch:subject <http://example.org/vehicles#Vehicle>
        ]
        [
            a manch:ClassFrame ;
            manch:clauses (
                [
                    a manch:ClassSubClassOfClause ;
                    manch:items (
                        [
                            a manch:ClauseItem ;
                            manch:value <http://example.org/vehicles#Vehicle>
                        ]
                    )
                ]
            ) ;
            manch:subject <http://example.org/vehicles#Car>
        ]
        [
            a manch:ObjectPropertyFrame ;
            manch:clauses () ;
            manch:subject <http://example.org/vehicles#hasWheelCount>
        ]
    ) ;
    manch:misc () ;
    manch:prefixes (
        [
            a ma

## 14. Edit the document by editing the RDF

Splice a new item into `Car`'s `SubClassOf:` clause's own `manch:items` list, adding `hasWheelCount some xsd:integer` as a second superclass expression - directly as RDF graph surgery, no text-level find/replace anywhere.

In [16]:
from pyoxigraph import BlankNode
from rdflib import RDF
from rdflib.namespace import XSD
from starlayer.starontology.manchester import MANCH

sub_class_of_clause = next(graph.subjects(RDF.type, MANCH.ClassSubClassOfClause))
items_list = next(graph.objects(sub_class_of_clause, MANCH.items))

# manch:Some's own onProperty/filler shape (see manchester-ast-ontology.ttl)
some_expr = BlankNode()
graph.add((some_expr, RDF.type, MANCH.Some))
graph.add((some_expr, MANCH.onProperty, EX.hasWheelCount))
graph.add((some_expr, MANCH.filler, XSD.integer))

new_item = BlankNode()
graph.add((new_item, RDF.type, MANCH.ClauseItem))
graph.add((new_item, MANCH.value, some_expr))


def list_cells(head):
    """The cells of an rdf:List, from its head to its last cell."""
    cell = head
    while cell.value != str(RDF.nil):
        yield cell
        cell = next(graph.objects(cell, RDF.rest))


# append new_item to the manch:items rdf:List: a new cell after the last one
last = list(list_cells(items_list))[-1]
new_cell = BlankNode()
graph.add((new_cell, RDF.first, new_item))
graph.add((new_cell, RDF.rest, RDF.nil))
graph.remove((last, RDF.rest, RDF.nil))
graph.add((last, RDF.rest, new_cell))
print('items after edit:', len(list(list_cells(items_list))))

items after edit: 2


## 15. Render back to Manchester Syntax text

`manchester_tree_to_text()` renders the (now-edited) RDF graph straight back to real Manchester Syntax text - it's the literal inverse of the parser's own tuple vocabulary, so it can render `Car`'s edited `SubClassOf:` clause with both items on one line, exactly as if that had been typed by hand. Editing happens only on the RDF graph itself (the plain triple surgery in section 14 above) - there's no intermediate Python object exposed to edit instead.

In [17]:
from starlayer.starontology.manchester import manchester_tree_to_text

edited_text = manchester_tree_to_text(graph)
print(edited_text)


Prefix: : <http://example.org/vehicles#>

Class: <http://example.org/vehicles#Vehicle>

Class: <http://example.org/vehicles#Car>
    SubClassOf: <http://example.org/vehicles#Vehicle>, <http://example.org/vehicles#hasWheelCount> some <http://www.w3.org/2001/XMLSchema#integer>

ObjectProperty: <http://example.org/vehicles#hasWheelCount>



## 16. Prove the edit was structural

Compile *both* the original and the edited text through the real, unmodified `parse_manchester()` and compare the resulting OWL-semantic triples. The edited version should introduce new triples into this compiled OWL RDF - the new `owl:someValuesFrom` restriction on `Car` - and so should genuinely differ, proving the graph-surgery edit changed real ontology semantics, not just cosmetic layout.

In [18]:
from starlayer.graph.compare import isomorphic

edited_triples = parse_manchester(edited_text)

print('original triple count:', len(original_triples))
print('edited triple count:  ', len(edited_triples))
print('isomorphic (should be False - the edit is a real semantic change):', isomorphic(original_triples, edited_triples))

original triple count: 4
edited triple count:   8
isomorphic (should be False - the edit is a real semantic change): False


## 17. Validate the edited document, with RDFS reasoning over the ontology

`manchester.manchester_validate()` runs the shapes graph against a data graph with `inference="rdfs"` and the `manch:` ontology - real RDFS reasoning before SHACL validation, the same mechanism `starlayer.sparql.validate_query()` uses, and for the same reason: it's what lets a dispatch shape like `manch:ClassExpressionShape` check a single abstract superclass instead of enumerating every concrete tag.

Then deliberately corrupt one triple - retype the spliced-in `manch:Some` restriction as something that isn't a recognized class-expression tag at all - and confirm a real SHACL violation is reported, not a silent pass.

In [19]:
from starlayer.starontology import manchester

conforms, results_graph, results_text = manchester.manchester_validate(graph)
print('conforms:', conforms)

conforms: True


In [20]:
graph.remove((some_expr, RDF.type, MANCH.Some))

conforms, results_graph, results_text = manchester.manchester_validate(graph)
print('conforms after corruption (should be False):', conforms)
print('report graph is an', type(results_graph).__name__)
print(results_text)

conforms after corruption (should be False): False
report graph is an OxGraph
Validation Report
Conforms: False

Pre-flight problems (asserted graph, checked before inference):
  - node has manch:filler, onProperty but no asserted concrete manch: type (node e697e72c14134f2b5dc05c7a96a6b4ff)

Validation Report
Conforms: False
Results (1):
Constraint Violation in NodeConstraintComponent (http://www.w3.org/ns/shacl#NodeConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:node manch:ClassExpressionItemListShape ; sh:path manch:items ]
	Focus Node: [ manch:items ( [ manch:value <http://example.org/vehicles#Vehicle> ; rdf:type manch:ClauseItem, rdfs:Resource ] [ manch:value [ manch:filler xsd:integer ; manch:onProperty <http://example.org/vehicles#hasWheelCount> ; rdf:type rdfs:Resource ] ; rdf:type manch:ClauseItem, rdfs:Resource ] ) ; rdf:type manch:ClassSubClassOfClause, manch:Clause, rdfs:Resource ]
	Value Node: ( [ manch:value <http://example.org/vehicles#Vehicle> ; rdf:typ

## Known limitations

One construct is deliberately unimplemented: SWRL `Rule:`. It raises `ManchesterSyntaxError` rather than being silently dropped or mis-parsed. (Annotations on the six top-level Misc axioms from section 10, `EquivalentClasses:` and friends, are supported; a one-member `EquivalentClasses:` is rejected, because the grammar needs two or more.) Full conformance tracking — including what's matched, what's not, and why — lives in [manchester_syntax_gap_analysis.md](../../packages/starlayer/docs/graph/manchester_syntax_gap_analysis.md).


In [21]:
from starlayer.graph.parsers.errors import ManchesterSyntaxError

try:
    show("""
    Rule: Person(?x)
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")

try:
    show("""
    Class: A
    Class: B
    EquivalentClasses: A
    """)
except ManchesterSyntaxError as e:
    print(f"Rejected as expected: {e.why}")


Rejected as expected: expected a Prefix:/Ontology:/frame/Misc-axiom keyword, got 'Rule:'
Rejected as expected: EquivalentClasses: needs at least two members, got 1
